In [1]:
import sys, os, tempfile, timeit, pickle, inspect
from dsc.dsc_io import load_dsc as __load_dsc__, source_dirs as __source_dirs__
import numpy as np

sys.path.append("/gpfs/commons/home/sbanerjee/work/npd/lrma-dsc/dsc/functions")
from comparison_metrics import (
    standardize,
    coupled_procrustes_per_factor_scaling,
    root_mean_squared_error,
    peak_signal_to_noise_ratio,
    adjusted_mutual_information_score
)

In [4]:
simres = __load_dsc__(['/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/blockdiag_p/blockdiag_p_1.pkl','/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/flashier/blockdiag_p_1_identical_1_flashier_1.rds'])

In [5]:
simres.keys()

dict_keys(['Z', 'Zmask', 'effect_size_obs', 'effect_size_true', 'Ltrue', 'Ftrue', 'Mtrue', 'Ctrue', 'nsample', 'DSC_DEBUG', 'L_est', 'F_est', 'S2'])

In [6]:
F = simres['F_est']
L = simres['L_est']
Ftrue = simres['Ftrue']
Ltrue = simres['Ltrue']
labels = simres['Ctrue']
Ztrue = simres['Z']

aligned = coupled_procrustes_per_factor_scaling(Ltrue, Ftrue, L, F, dim_policy="zerofill")
Lt = aligned["L_true"]
Ft = aligned["F_true"]
La = aligned["L_aligned"]
Fa = aligned["F_aligned"]

In [7]:
Zt = standardize(Ltrue @ Ftrue.T)
Za = standardize(L @ F.T)

root_mean_squared_error(Zt, Za)

0.37260870918877204

In [8]:
Zt = Ltrue @ Ftrue.T
Za = L @ F.T

root_mean_squared_error(Zt, Za)

2.2607976270683756

In [9]:
Zt = Ltrue @ Ftrue.T
Za = L @ F.T
global_scale = np.sum(Zt * Za) / np.sum(Za ** 2)
root_mean_squared_error(Zt, global_scale * Za)

0.004837503026148258

In [32]:
Zt

array([[ 0.03217388,  0.01346469,  0.00356469, ...,  0.01296791,
        -0.01926241, -0.0054789 ],
       [ 0.0061947 ,  0.00676196,  0.00537825, ...,  0.00103078,
         0.00198027, -0.01826607],
       [ 0.00538006,  0.01437222,  0.00940027, ...,  0.00563223,
        -0.01125115, -0.0145367 ],
       ...,
       [-0.00537736, -0.00571403, -0.0084934 , ..., -0.02077178,
         0.03350434, -0.00726771],
       [ 0.00602554,  0.01212874,  0.02249626, ..., -0.01100464,
         0.01261926, -0.01281982],
       [ 0.00575924,  0.01549025,  0.02977025, ..., -0.01393819,
         0.01645002, -0.01879696]])

In [33]:
global_scale * Za

array([[ 0.03356764,  0.006324  ,  0.00623025, ...,  0.00608248,
        -0.01316163,  0.00172157],
       [ 0.00440259,  0.00294096,  0.00173867, ..., -0.00061781,
         0.00165991, -0.0083506 ],
       [ 0.01198219,  0.02072298,  0.00825671, ...,  0.0118947 ,
        -0.01184025, -0.00968488],
       ...,
       [-0.00577985, -0.00187674, -0.00665133, ..., -0.01586498,
         0.02572651, -0.00469118],
       [ 0.00255276,  0.01726696,  0.02021363, ..., -0.00721264,
         0.01185104, -0.01715977],
       [ 0.00621432,  0.01802937,  0.02710486, ..., -0.01208069,
         0.01404398, -0.02367894]])

In [22]:
root_mean_squared_error(Lt, global_scale * La)

0.03408493553903409

In [23]:
root_mean_squared_error(Ft, Fa)

0.015421217251120185

In [29]:
root_mean_squared_error(Ft, Fa / global_scale)

6.676134032556894

In [28]:
np.allclose((global_scale * La) @ (Fa / global_scale).T, L @ F.T)

True